In [46]:
import os
from dotenv import load_dotenv
from langchain_groq import ChatGroq
from langchain_huggingface import HuggingFaceEmbeddings
import chromadb
from langchain_community.vectorstores import Chroma
from langchain_community.document_loaders import PyMuPDFLoader
from langchain_experimental.text_splitter import SemanticChunker
from crewai import Agent, Task, Crew, Process

# Load environment variables
load_dotenv(override=True)

# --- 1. LLM & EMBEDDINGS (Groq + CUDA) ---
llm = ChatGroq(
    model_name="llama3-70b-8192", 
    groq_api_key=os.getenv("GROQ_API_KEY"),
    temperature=0
)

embeddings = HuggingFaceEmbeddings(
    model_name="BAAI/bge-small-en-v1.5",
    model_kwargs={'device': 'cuda'} )

# --- 3. CUSTOM CREWAI TOOLS ---
from langchain_core.tools import Tool


# --- 2. VECTOR DB SETUP (Persistence) ---
persist_directory = r"C:\Users\user\Desktop\RAG Projects\Legal RAG 1\vector_database"
vectorstore = Chroma(persist_directory=persist_directory, embedding_function=embeddings)


Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

BertModel LOAD REPORT from: BAAI/bge-small-en-v1.5
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


### **Vector Store Configuration**

In [47]:
client = chromadb.PersistentClient(path=r"C:\Users\user\Desktop\RAG Projects\Legal RAG 1\vector_database")

vector_store = Chroma(
    client=client,
    collection_name="legal_knowledge",
    embedding_function=embeddings,
)

print(f"✅ LangChain Chroma connection established at: {client.persist_directory}")

AttributeError: 'Client' object has no attribute 'persist_directory'

In [3]:
from langchain_community.document_loaders import PyMuPDFLoader
from llama_index.core import Document

# 1. Update File Path: Point to your specific PDF
file_path = r"C:\Users\user\Desktop\RAG Projects\Legal RAG 1\data\Indian_Constitution.pdf"

# 2. Load the PDF using LangChain's specialized loader
# PyMuPDF is excellent for maintaining the layout of legal documents
loader = PyMuPDFLoader(file_path)
vector_docs = loader.load()

# 3. Conversion Step: Transform LangChain format to LlamaIndex format
# This is the "Compatibility Bridge" required for your Graph and Vector Index
graph_docs = [Document(text=d.page_content, metadata={"page_label": str(d.metadata.get("page", "Unknown"))}) for d in vector_docs]

print(f"Successfully loaded and converted {len(graph_docs)} pages from the Indian Constitution.")

Successfully loaded and converted 402 pages from the Indian Constitution.


In [4]:
print(vector_docs[0])

page_content='£ÉÉ®iÉ BÉEÉ ºÉÆÉÊ´ÉvÉÉxÉ 
[1
, 2024
]
THE CONSTITUTION OF INDIA
[As on 1st May, 2024] 
2024
GOVERNMENT OF INDIA
MINISTRY OF LAW AND JUSTICE
LEGISLATIVE DEPARTMENT, OFFICIAL LANGUAGES WING' metadata={'producer': 'iLovePDF', 'creator': '', 'creationdate': '', 'source': 'C:\\Users\\user\\Desktop\\RAG Projects\\Legal RAG 1\\data\\Indian_Constitution.pdf', 'file_path': 'C:\\Users\\user\\Desktop\\RAG Projects\\Legal RAG 1\\data\\Indian_Constitution.pdf', 'total_pages': 402, 'format': 'PDF 1.7', 'title': '', 'author': '', 'subject': '', 'keywords': '', 'moddate': '2024-07-01T11:20:33+00:00', 'trapped': '', 'modDate': 'D:20240701112033Z', 'creationDate': '', 'page': 0}


In [5]:
print(graph_docs[0])

Doc ID: cd313e21-324c-48d9-bbd4-ae33e832ce96
Text: £ÉÉ®iÉ BÉEÉ ºÉÆÉÊ´ÉvÉÉxÉ  [1 , 2024 ] THE CONSTITUTION OF INDIA
[As on 1st May, 2024]  2024 GOVERNMENT OF INDIA MINISTRY OF LAW AND
JUSTICE LEGISLATIVE DEPARTMENT, OFFICIAL LANGUAGES WING


In [6]:
# articles short
#Pages 3 to 31 contains the  introduction
articles_short = vector_docs[3:31]
print(articles_short[0])

page_content='THE CONSTITUTION OF INDIA
____________                                                                    
CONTENTS
__________
                                                                                          
PREAMBLE
PART I 
THE UNION AND ITS TERRITORY
ARTICLES 
1.
Name and territory of the Union.
  2. 
Admission or establishment of new States. 
[2A.         Sikkim to be associated with the Union.—Omitted.]
3.
Formation of new States and alteration of areas, boundaries or 
names of existing States.
4.
Laws made under articles 2 and 3 to provide for the amendment of 
the First and the Fourth Schedules and supplemental, incidental 
and consequential matters.
PART II
CITIZENSHIP
5.
Citizenship at the commencement of the Constitution.
6.
Rights of citizenship of certain persons who have migrated to 
India from Pakistan.
7.
Rights of citizenship of certain migrants to Pakistan.
8.
Rights of citizenship of certain persons of Indian origin residing    
outside India.
9

In [7]:
# preamble
preamble = vector_docs[31]
preamble

Document(metadata={'producer': 'iLovePDF', 'creator': '', 'creationdate': '', 'source': 'C:\\Users\\user\\Desktop\\RAG Projects\\Legal RAG 1\\data\\Indian_Constitution.pdf', 'file_path': 'C:\\Users\\user\\Desktop\\RAG Projects\\Legal RAG 1\\data\\Indian_Constitution.pdf', 'total_pages': 402, 'format': 'PDF 1.7', 'title': '', 'author': '', 'subject': '', 'keywords': '', 'moddate': '2024-07-01T11:20:33+00:00', 'trapped': '', 'modDate': 'D:20240701112033Z', 'creationDate': '', 'page': 31}, page_content='THE CONSTITUTION OF INDIA \nPREAMBLE\nWE, THE PEOPLE OF INDIA, having solemnly resolved to constitute \nIndia into a \n1[SOVEREIGN SOCIALIST SECULAR DEMOCRATIC \nREPUBLIC] and to secure to all its citizens:\nJUSTICE, social, economic and political;\n \nLIBERTY of thought, expression, belief, faith and worship;\nEQUALITY of status and of opportunity;\nand to promote among them all\nFRATERNITY assuring the dignity of the individual and the 2[unity \nand integrity of the Nation];\nIN OUR CONS

In [8]:
articles = vector_docs[32:283]

In [9]:
print(articles[0].page_content)

2
PART I
THE UNION AND ITS TERRITORY
1. Name and territory of the Union.—(1) India, that is Bharat, 
shall be a Union of States.
1[(2) The States and the territories thereof shall be as specified in 
the First Schedule.]
(3) The territory of India shall comprise—
(a) the territories of the States; 
2[(b) the Union territories specified in the First Schedule; 
and]
(c) such other territories as may be acquired.
2. Admission or establishment of new States.—Parliament may 
by law admit into the Union, or establish, new States on such terms and 
conditions as it thinks fit.
3[2A. [Sikkim to be associated with the Union.] —Omitted by the 
Constitution (Thirty-sixth Amendment) Act, 1975, s. 5 (w.e.f. 26-4-1975).]
3. Formation of new States and alteration of areas, 
boundaries or names of existing States.—Parliament may by law—
(a) form a new State by separation of territory from any 
State or by uniting two or more States or parts of States or by 
uniting any territory to a part of any State

In [10]:
##code to remove headers  and footers from pdf for vector db
import re
for art in articles:
  art.page_content = re.split(r'_{2,}', art.page_content)[0].strip()

In [11]:
from llama_index.core import Document

# 1. Combine the raw LangChain objects
all_vector_docs_raw = articles_short + [preamble] + articles

# 2. Convert LangChain objects to LlamaIndex objects
all_vector_docs = [Document(text=d.page_content, metadata={"page_label": str(d.metadata.get("page", "Unknown"))}) for d in all_vector_docs_raw]
# 3. Now the LlamaIndex methods will work
for doc in all_vector_docs:
    # Now .get_content() exists because these are LlamaIndex objects
    raw_text = doc.get_content()
    cleaned = re.split(r'_{2,}', raw_text)[0].strip()
    doc.set_content(cleaned)

print(f"✅ Successfully converted and cleaned {len(all_vector_docs)} pages.")

✅ Successfully converted and cleaned 280 pages.


In [12]:
##code to remove headers  and footers from pdf for vector db
import re

# 1. Define the cleaning function to reuse it
def clean_legal_document(doc_text):
    if doc_text:
        # 1. Remove common Diglot/Hindi garbled characters (Non-ASCII)
        # This keeps only English letters, numbers, and standard punctuation
        doc_text = "".join(i for i in doc_text if ord(i) < 128)
        
        # 2. Your existing underscore removal logic
        return re.split(r'_{2,}', doc_text)[0].strip()
    return ""

# 2. Clean the entire Graph Document set
# We do NOT slice here; we want the graph to see the Preamble, Articles, and Schedules
for doc in graph_docs:
    # Note: LlamaIndex Document uses .text instead of .page_content
    # If you just converted them from LangChain, ensure you target the right attribute
    original_text = doc.get_content()
    cleaned_text = clean_legal_document(original_text)
    
    # This is the line that fixes your AttributeError
    doc.set_content(cleaned_text)

print(f"Preprocessing complete: {len(graph_docs)} pages cleaned for Graph Ingestion.")

Preprocessing complete: 402 pages cleaned for Graph Ingestion.


In [13]:
# --- STEP 1: LOAD AND TAG SCHEDULES ---

# 1. Slice the schedules from your original vector_docs (LangChain objects)
# Based on your previous notebook: 283 to 381
schedules_raw = vector_docs[283:381]

# 2. Convert and tag them as "Schedules"
# We add a specific 'category' and 'type' to the metadata
schedules_docs = [
    Document(
        text=d.page_content, 
        metadata={
            "page_label": str(d.metadata.get("page", "Unknown")),
            "section_type": "Schedule",
            "document_part": "Schedules of the Constitution"
        }
    ) for d in schedules_raw
]

# --- STEP 2: UNIFIED CLEANING ---

# Clean the Schedules just like you did for the Articles
for doc in schedules_docs:
    raw_text = doc.get_content()
    # Remove Hindi/Diglot noise and footer underscores
    cleaned = "".join(i for i in raw_text if ord(i) < 128)
    cleaned = re.split(r'_{2,}', cleaned)[0].strip()
    doc.set_content(cleaned)

# --- STEP 3: MERGE INTO YOUR FINAL LISTS ---

# Add the schedules to your existing vector list
# This ensures vector_chapter will now contain Preamble + Articles + Schedules
all_vector_docs = all_vector_docs + schedules_docs

# Note: Since graph_docs was already a full load, ensure its metadata 
# is also updated if you want specific 'section_type' tags there.

In [14]:
# --- STEP 4: ENRICH AND CLEAN GRAPH DOCUMENTS ---

for i, doc in enumerate(graph_docs):
    # 1. Access the raw text
    raw_text = doc.get_content()
    
    # 2. Clean the text (Remove Hindi and Footers)
    cleaned = "".join(char for char in raw_text if ord(char) < 128)
    cleaned = re.split(r'_{2,}', cleaned)[0].strip()
    doc.set_content(cleaned)
    
    # 3. Apply Metadata Tags based on Page Ranges
    # Note: We use the page numbers to identify the section type
    page_num = int(doc.metadata.get("page_label", -1))
    
    if 32 <= page_num <= 282:
        doc.metadata["section_type"] = "Article"
        doc.metadata["document_part"] = "Main Constitution"
    elif page_num >= 283:
        doc.metadata["section_type"] = "Schedule"
        doc.metadata["document_part"] = "Schedules"
    elif page_num == 31:
        doc.metadata["section_type"] = "Preamble"
        doc.metadata["document_part"] = "Introduction"

In [39]:
from llama_index.core.node_parser import SemanticSplitterNodeParser

# Initialize the Semantic Splitter
splitter = SemanticSplitterNodeParser(
    buffer_size=1, 
    breakpoint_percentile_threshold=90, 
    embed_model=Settings.embed_model
)

# Process documents into semantically coherent nodes
# for graph_docs
graph_nodes = splitter.get_nodes_from_documents(graph_docs)

# for vector_docs

vector_chapter = splitter.get_nodes_from_documents(all_vector_docs)

c:\Users\user\AppData\Local\Programs\Python\Python310\lib\site-packages\llama_index\core\schema.py:332: RuntimeWarning: coroutine 'SimpleLLMPathExtractor._aextract' was never awaited
  if mode == MetadataMode.NONE:


In [40]:
def inspect_nodes(nodes, name="Nodes"):
    print(f"--- Inspection Report: {name} ---")
    print(f"Total Nodes Created: {len(nodes)}\n")
    
    # Check the first 3 nodes to see the start of the document (Preamble/Intro)
    for i, node in enumerate(nodes[:3]):
        print(f"NODE {i} (Page {node.metadata.get('page_label', 'Unknown')}):")
        # Print first 200 and last 200 characters to see the 'breaks'
        text = node.get_content().replace('\n', ' ')
        print(f"START: {text[:200]}...")
        print(f"END:   ...{text[-200:]}")
        print("-" * 30)

# 1. Get output for Graph Nodes (Holistic)
inspect_nodes(graph_nodes, "Graph Database Nodes")

# 2. Get output for Vector Nodes (Slices)
inspect_nodes(vector_chapter, "Vector Database Nodes")

--- Inspection Report: Graph Database Nodes ---
Total Nodes Created: 1013

NODE 0 (Page 0):
START: i BE vx  [1 , 2024 ] THE CONSTITUTION OF INDIA [As on 1st May, 2024]  2024 GOVERNMENT OF INDIA MINISTRY OF LAW AND JUSTICE LEGISLATIVE DEPARTMENT, OFFICIAL LANGUAGES WING...
END:   ...i BE vx  [1 , 2024 ] THE CONSTITUTION OF INDIA [As on 1st May, 2024]  2024 GOVERNMENT OF INDIA MINISTRY OF LAW AND JUSTICE LEGISLATIVE DEPARTMENT, OFFICIAL LANGUAGES WING
------------------------------
NODE 1 (Page 1):
START: PREFACE This is the  sixth pocket size edition of the Constitution of  India in the diglot form. In this edition, the text of the  Constitution of India has been brought up-to-date by  incorporating t...
END:   ...ication to Jammu and Kashmir)  Order, 2019 and the declaration under article 370(3) of the  Constitution have been provided respectively in Appendix II and  Appendix III for reference. New Delhi; Dr. 
------------------------------
NODE 2 (Page 1):
START: Rajiv Mani, 1st May, 

### **Vector Graphs of Indian Constitution**

In [17]:
from llama_index.core import VectorStoreIndex

# 1. Create the Vector Index
# This uses the 'vector_chapter' nodes (805 total)
vector_index = VectorStoreIndex(
    vector_chapter, 
    storage_context=storage_context, 
    show_progress=True
)

# 2. Set the collection name to match your persistent storage
vector_index.set_index_id("legal_knowledge_base")

print("Vector Index successfully built and persisted to disk.")

Generating embeddings:   0%|          | 0/805 [00:00<?, ?it/s]

Vector Index successfully built and persisted to disk.


In [23]:
import os
from dotenv import load_dotenv

load_dotenv(override=True)

print(f"URI: [{os.getenv('NEO4J_URI2')}]")
print(f"User: [{os.getenv('NEO4J_USERNAME2')}]")

URI: [bolt://localhost:7687]
User: [neo4j]


In [41]:
import nest_asyncio
import os
from llama_index.llms.ollama import Ollama
from llama_index.graph_stores.neo4j import Neo4jPropertyGraphStore
from llama_index.core import PropertyGraphIndex, Settings

# 1. Apply the async fix for Jupyter
nest_asyncio.apply()

# 2. Configure Settings to use your local Ollama model
# Ensure you have run 'pip install llama-index-llms-ollama'
Settings.llm = Ollama(model="llama3:latest", request_timeout=600.0)

# 3. Establish connection to Neo4j
graph_store = Neo4jPropertyGraphStore(
    username=os.getenv("NEO4J_USERNAME2"),
    password=os.getenv("NEO4J_PASSWORD2"),
    url=os.getenv("NEO4J_URI2"),
    database="neo4j",
    refresh_schema=False 
)

# 4. Build the Index directly (No more batching or sleeping needed!)
print(f"🚀 Starting local indexing of {len(graph_nodes)} nodes using Ollama...")

graph_index = PropertyGraphIndex(
    graph_nodes,
    property_graph_store=graph_store,
    show_progress=True,
    embed_model=Settings.embed_model,
    num_workers=4  # <--- Add this to process 4 nodes at a time locally
)

print("✅ FINAL REPORT: Local Property Graph Index successfully built in Neo4j!")

🚀 Starting local indexing of 1013 nodes using Ollama...


Applying transformations:   0%|          | 0/2 [00:00<?, ?it/s]

KeyboardInterrupt: 